In [0]:
# log_workflow_info

import requests
import json
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql.functions import col, from_json, schema_of_json, to_json, from_utc_timestamp
from pyspark.sql.types import StructType, StructField, StringType, LongType, ArrayType, IntegerType, TimestampType


In [0]:
def get_duration_components(duration_ms):
    duration_seconds = duration_ms / 1000
    minutes = int(duration_seconds // 60)
    seconds = int(duration_seconds % 60 )
    return f"{minutes} min {seconds} sec"

In [0]:
# Setup
# dbutils.widgets.text("job_id", "")
# dbutils.widgets.text("job_run_id", "")
job_id = dbutils.widgets.get("job_id")
job_run_id = dbutils.widgets.get("job_run_id")
task_name = dbutils.widgets.get("task_name")

## Adding logging variables 

In [ ]:
import yaml

current_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()

# Get the directory containing the notebook (remove the notebook filename)
parameter_directory = '/'.join(current_notebook_path.split('/')[:-1])

# Build the CSV file path
file_path = f"/Workspace{parameter_directory}/util/parameters.yml"

with open(file_path, "r") as file:
    config = yaml.safe_load(file)

In [0]:
# logging table variable
logging_catalog = config["logging_catalog"]
logging_schema = config["logging_schema"]
log_table = config["job_logs_table"]

In [0]:
# Databricks REST API
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
workspace_url = "https://" + spark.conf.get("spark.databricks.workspaceUrl")

headers = {
    "Authorization": f"Bearer {token}"
}

# Get job details
job_resp = requests.get(
    f"{workspace_url}/api/2.1/jobs/get?job_id={job_id}",
    headers=headers
)

# Get run details
run_resp = requests.get(
    f"{workspace_url}/api/2.1/jobs/runs/get?run_id={job_run_id}",
    headers=headers
)

job_data = job_resp.json()
job_run_data = run_resp.json()

In [0]:
tasks_data = []
for task in job_run_data.get("tasks"):
    task_run_id = task.get("run_id")
    task_key = task.get("task_key")
    if task_key == task_name:
        continue
    depends_on = task.get("depends_on", None)
    pipeline_task = task.get("pipeline_task",{}).get("pipeline_id",None)
    task_time = get_duration_components(task.get("execution_duration"))
    task_run_status = task.get("state", {}).get("result_state")
    task_run_status_code = task.get("status", {}).get("termination_details",{}).get("code")
    task_run_status_message = task.get("state",{}).get("state_message")
    
    task_data_dict = {"task_run_id": task_run_id, "task_name": task_key , "task_depends_on": depends_on,"Pipeline_task": pipeline_task, "task_execution_time": task_time, "task_run_status": task_run_status, "task_run_status_code": task_run_status_code, "task_run_status_message": task_run_status_message}

    tasks_data.append(task_data_dict)

In [0]:
# get execution time only
job_run_execution_duration = sum([i.get('execution_duration') for i in job_run_data.get("tasks") if i.get("task_key") != task_name])

#create status columns
all_success = all(task['task_run_status'] == 'SUCCESS' for task in tasks_data)
all_success_code = all(task['task_run_status_code'] == 'SUCCESS' for task in tasks_data)
all_success_message = all(task['task_run_status_message'] == '' for task in tasks_data)

task_status_map = {task['task_name']: task['task_run_status'] for task in tasks_data}
task_status_code_map = {task['task_name']: task['task_run_status_code'] for task in tasks_data}

# Construct the values for the columns
job_run_status = {
    "Job_Run_Status": "success" if all_success else "Success_With_Failures",
    "Task_Run_status_details": f"{task_status_map}"
}

job_run_status_code = {
    "Job_Run_Status_Code": "Success" if all_success_code else "RunExecutionError",
    "Task_Run_status_code_Details": f"{task_status_code_map}"
}

job_run_status_message = None if all_success_message else 'Workload failed, see run output for details'

In [0]:
log_entry = Row(
                workflow_name = job_data.get("settings", {}).get("name"),
                job_id = job_run_data.get('job_id'),
                job_run_id = job_run_data.get('run_id'),
                start_time_job=datetime.fromtimestamp(job_run_data.get("start_time")/1000),
                end_time_job= datetime.fromtimestamp((job_run_data.get("start_time") + job_run_data.get("run_duration"))/1000),
                cluster_step_up_duration = get_duration_components(job_run_data.get("run_duration") - job_run_execution_duration), 
                job_run_execution_duration = get_duration_components(job_run_execution_duration), # execution time only
                total_job_run_duration = get_duration_components(job_run_data.get("run_duration")), # complete execution time for job
                run_page = job_run_data.get("run_page_url"),
                tasks = json.dumps(tasks_data),
                job_run_status = json.dumps(job_run_status),
                job_run_status_code = json.dumps(job_run_status_code),
                job_run_status_message = job_run_status_message,
                insert_time = datetime.now()
                )


In [0]:
schema = StructType([
    StructField("workflow_name", StringType(), True),
    StructField("job_id", LongType(), True),
    StructField("job_run_id", LongType(), True),
    StructField("start_time_job", TimestampType(), True),
    StructField("end_time_job", TimestampType(), True),
    StructField("cluster_step_up_duration", StringType(), True),
    StructField("job_run_execution_duration", StringType(), True),
    StructField("total_job_run_duration", StringType(), True),
    StructField("run_page", StringType(), True),
    StructField("tasks", StringType(), True),
    StructField("job_run_status", StringType(), True),
    StructField("job_run_status_code", StringType(), True),
    StructField("job_run_status_message", StringType(), True),
    StructField("insert_time", TimestampType(), True)
])

df = spark.createDataFrame([log_entry],schema)

structured_string_columns = ["tasks","job_run_status","job_run_status_code"]

timestamp_columns = ["start_time_job", "end_time_job", "insert_time"]
for col_name in timestamp_columns:
    df = df.withColumn(col_name, from_utc_timestamp(col_name, "Asia/Kolkata"))

# df.display()

In [0]:
# Function to convert string columns in a specific structure format to their actual data types
def convert_string_columns(df, columns):
    for column in columns:
        #print(column)
        if column in df.columns:
            json_schema = schema_of_json(df.select(column).first()[0])
            #print(json_schema)
            df = df.withColumn(column, from_json(col(column), json_schema))
    return df

# Specify the columns to convert
df_converted_structure = convert_string_columns(df, structured_string_columns)
#display(df_converted_structure)

workflow_name,job_id,job_run_id,start_time_job,end_time_job,cluster_step_up_duration,job_run_execution_duration,total_job_run_duration,run_page,tasks,job_run_status,job_run_status_code,job_run_status_message,insert_time
[dev eena_agrawal] [dev] dev_auto_dbx_proj_job,568890082522385,10418672417731,2025-05-13T18:00:33.855Z,2025-05-13T18:05:35.184Z,3 min 22 sec,1 min 39 sec,5 min 1 sec,https://dbc-d03e585a-0b01.cloud.databricks.com/?o=1207160196612479#job/568890082522385/run/10418672417731,"List(List(null, List(List(refresh_pipeline)), 0 min 43 sec, main_task, 862474904389817, SUCCESS, SUCCESS, ), List(ecb0c032-07e0-4af3-9905-35b0a0ae5e2d, List(List(notebook_task)), 0 min 21 sec, refresh_pipeline, 937377029911932, SUCCESS, SUCCESS, ), List(null, null, 0 min 35 sec, notebook_task, 1110386845516405, SUCCESS, SUCCESS, ))","List(success, {'main_task': 'SUCCESS', 'refresh_pipeline': 'SUCCESS', 'notebook_task': 'SUCCESS'})","List(Success, {'main_task': 'SUCCESS', 'refresh_pipeline': 'SUCCESS', 'notebook_task': 'SUCCESS'})",null,2025-05-13T18:23:22.965251Z


In [0]:
df_converted_structure.write.mode("append").saveAsTable(f"{logging_catalog}.{logging_schema}.{log_table}")